# LLM Output Evaluator

Compare multiple language-model responses using a transparent weighted rubric.


In [ ]:
import json
import pandas as pd
from pathlib import Path


In [ ]:
CRITERIA = {
    "relevance": 0.20,
    "coherence": 0.15,
    "factual_consistency": 0.20,
    "tone": 0.10,
    "naturalness": 0.15,
    "instruction_alignment": 0.20,
}


In [ ]:
data = {
    "instruction": "Explain overfitting to a beginner in under 80 words.",
    "responses": [
        {"id":"A","text":"Overfitting happens when a model memorizes the training examples too closely. It performs well on familiar data but poorly on new data. It is like studying only the exact answers to a practice test instead of learning the ideas. Techniques such as cross-validation, regularization, and simpler models can help."},
        {"id":"B","text":"Overfitting is when predictive variance emerges from excessive parameterization and empirical risk minimization."},
        {"id":"C","text":"A model overfits when it learns noise and details from training data that do not generalize. It may score highly during training but fail on unseen examples."}
    ]
}


In [ ]:
def score_response(text: str, instruction: str) -> dict[str, float]:
    words = text.split()
    lower = text.lower()
    return {
        "relevance": 5.0 if "overfit" in lower else 2.0,
        "coherence": 5.0 if len(words) >= 15 and text.endswith((".", "!", "?")) else 3.0,
        "factual_consistency": 5.0 if any(k in lower for k in ["training", "new data", "unseen", "generalize"]) else 3.0,
        "tone": 5.0 if not any(k in lower for k in ["variance emerges", "empirical risk"]) else 2.0,
        "naturalness": 5.0 if len(words) <= 80 else 3.0,
        "instruction_alignment": 5.0 if len(words) <= 80 and any(k in lower for k in ["like", "beginner", "new data", "unseen"]) else 3.0,
    }


In [ ]:
rows = []
for item in data["responses"]:
    scores = score_response(item["text"], data["instruction"])
    weighted = sum(scores[k] * CRITERIA[k] for k in CRITERIA)
    rows.append({"response_id": item["id"], **scores, "weighted_score": round(weighted, 2), "text": item["text"]})

result = pd.DataFrame(rows).sort_values("weighted_score", ascending=False)
result


In [ ]:
result.to_csv("evaluation_report.csv", index=False)
